# E-commerce Customer Analytics & Segmentation

**Research question:** What characteristics distinguish repeat/high-value customers from one-time customers?

This notebook documents the complete analytical workflow used on an anonymized e-commerce order dataset.

## 1. Setup and Data Loading

The cleaned customer/order dataset is intentionally kept private and is excluded from GitHub.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../data/project1_clean_core.csv')
df.head()

## 2. Basic Data Validation

In [ ]:
df.shape
df.info()
df.isna().sum().sort_values(ascending=False).head(15)

## 3. Exploratory Data Analysis

The analysis examines revenue over time, sales channels, payment behaviour and order status.

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
monthly = df.groupby(df['date'].dt.to_period('M'))['sell_price_total'].sum()
monthly

## 4. RFM Analysis

In [ ]:
reference_date = df['date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    last_purchase=('date', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('sell_price_total', 'sum')
).reset_index()

rfm['recency'] = (reference_date - rfm['last_purchase']).dt.days
rfm = rfm[['customer_id', 'recency', 'frequency', 'monetary']]
rfm.head()

## 5. Transform and Scale RFM Features

Log transformation reduces the influence of extreme monetary values before standardization.

In [ ]:
features = ['recency', 'frequency', 'monetary']
X = np.log1p(rfm[features])
X = StandardScaler().fit_transform(X)

## 6. Select Number of Clusters

In [ ]:
results = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    results.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X, labels)
    })

selection = pd.DataFrame(results)
selection

### Final choice: 6 clusters

Six clusters were selected as a practical business segmentation. The decision considers both clustering quality and interpretability; it is not presented as the only mathematically possible solution.

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42, n_init=20)
rfm['cluster'] = kmeans.fit_predict(X)
rfm.groupby('cluster')[features].agg(['count', 'mean'])

## 7. Segment Interpretation

The final clusters are interpreted using recency, frequency and monetary behaviour. Names such as 'high-value one-time' and 'repeat purchasers' are business labels, not supervised target classes.

## 8. Business Recommendations

- High-value one-time customers → retention and cross-sell campaigns
- Repeat purchasers → loyalty and referral programs
- Recent potential customers → second-purchase campaigns
- Older low-value customers → low-cost reactivation
- Minimal-value customers → automated, low-cost communication


## 9. Limitations

- Source data originated from a PDF export.
- Some malformed/incomplete records were excluded.
- The observation period is limited.
- Segments describe behaviour but do not establish causal reasons for behaviour.
